# Version 1 — Réseau à une couche cachée
**Responsable : DIOP Alioune** · relecture : FALL Madjiguène

Parties 1.3, 1.4 et 2 du sujet, puis évaluation finale (unique) sur le test de la référence et du MLP retenu.

Temps d'exécution indicatif sur CPU : 15–25 min (le plus long est la grille de pas d'apprentissage).
Pour un essai rapide du notebook entier : exécuter avec la variable d'environnement `HAR_FAST=1`.

## 1. Imports

In [ ]:
import sys, os, json, time
sys.path.append("../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from har import config, data, models, training, metrics, plots

config.FIG_DIR.mkdir(parents=True, exist_ok=True); config.LOG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=0.9)
training.set_seed(0)
print("device :", training.DEVICE, "| torch", torch.__version__)

FAST = os.environ.get("HAR_FAST") == "1"
EPOCHS = 8 if FAST else 60          # entraînements principaux
EPOCHS_LR = 4 if FAST else 20       # grille de pas d'apprentissage
ACTS = ["relu", "sigmoid", "tanh"]
OPTS = ["sgd", "sgd_momentum", "adam"]
LR_GRID = [1e-3, 1e-2, 1e-1] if FAST else [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
SEEDS = config.SEEDS
CRIT = nn.CrossEntropyLoss()

def ms(v):
    """'moyenne ± écart-type' d'une liste de scores."""
    return metrics.mean_std(v)

def last_k(h, key="val_f1", k=3):
    """Score de fin d'entraînement, lissé sur les k dernières époques (moins sensible au bruit)."""
    return float(np.mean(h[key][-k:]))

## 2. Chargement et exploration des données
Résumé : l'exploration complète est dans `00_exploration.ipynb`.

In [ ]:
df = data.load_train()
print(df.shape, "|", df[config.SUBJECT_COL].nunique(), "volontaires")
df[config.TARGET_COL].value_counts().reindex(config.CLASSES)

## 3. Découpage apprentissage / validation et DataLoaders
La validation est formée de **5 volontaires entiers** de `train.csv` (graine figée `config.SPLIT_SEED`, donc le même
découpage pour toute l'équipe). Le `StandardScaler` est ajusté sur l'apprentissage uniquement.

In [ ]:
df_tr, df_va, val_subjects = data.split_by_subject(df)
assert not set(df_tr[config.SUBJECT_COL]) & set(df_va[config.SUBJECT_COL])
print("volontaires de validation :", val_subjects)
print(f"apprentissage : {len(df_tr)} fenêtres · validation : {len(df_va)} fenêtres ({len(df_va)/len(df):.1%})")

X_tr, y_tr, X_va, y_va, scaler = data.prepare(df_tr, df_va)
train_loader, val_loader = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
IN_DIM = X_tr.shape[1]
pd.DataFrame({"train": np.bincount(y_tr), "val": np.bincount(y_va)}, index=config.CLASSES)

### 3.1 Pourquoi des volontaires entiers ? (P1.3)
Les fenêtres font 2,56 s avec 50 % de recouvrement : deux fenêtres consécutives d'un même volontaire partagent la moitié
de leurs mesures. Avec un découpage aléatoire des lignes, presque chaque fenêtre de validation a une « jumelle » dans
l'apprentissage. On le vérifie : distance de chaque fenêtre de validation à sa plus proche voisine d'apprentissage.

In [ ]:
df_tr_rand, df_va_rand = data.split_random_rows(df, val_frac=len(df_va) / len(df))
nn_stats = {}
for name, (a, b) in {"lignes aléatoires": (df_tr_rand, df_va_rand), "par volontaire": (df_tr, df_va)}.items():
    Xa, ya, Xb, yb, _ = data.prepare(a, b)
    dist, idx = NearestNeighbors(n_neighbors=1).fit(Xa).kneighbors(Xb)
    same_subject = (a[config.SUBJECT_COL].to_numpy()[idx[:, 0]] == b[config.SUBJECT_COL].to_numpy()).mean()
    nn_stats[name] = {"distance médiane au plus proche voisin": float(np.median(dist)),
                      "plus proche voisin = même volontaire": f"{same_subject:.1%}"}
pd.DataFrame(nn_stats)

### 3.2 Découpage aléatoire vs par volontaire : même modèle, deux estimations (P1.3)
On entraîne la régression logistique et le MLP [64] (ReLU, Adam, 3 graines) avec chaque découpage.
Les modèles sont conservés : en section 7, on comparera ces estimations à la performance réelle sur le test.

In [ ]:
split_cmp, split_models = {}, {}
for name, (a, b) in {"lignes aléatoires": (df_tr_rand, df_va_rand), "par volontaire": (df_tr, df_va)}.items():
    Xa, ya, Xb, yb, sc = data.prepare(a, b)
    lr_model = LogisticRegression(max_iter=3000).fit(Xa, ya)
    split_cmp[(name, "régression logistique")] = metrics.compute_metrics(yb, lr_model.predict(Xb))

    def build(seed, Xa=Xa, ya=ya, Xb=Xb, yb=yb):
        tl, vl = data.make_loaders(Xa, ya, Xb, yb, seed=seed)
        m = models.MLP(IN_DIM, [64], activation="relu")
        return m, tl, vl, training.make_optimizer("adam", m.parameters(), lr=1e-3)
    res = training.run_seeds(build, seeds=SEEDS, keep_models=True, epochs=EPOCHS)
    split_cmp[(name, "MLP [64]")] = {"accuracy": res["summary"]["val_acc"]["mean"],
                                     "f1_macro": res["summary"]["val_f1"]["mean"]}
    split_models[name] = {"scaler": sc, "logreg": lr_model, "mlps": [r["model"] for r in res["runs"]]}

split_val = pd.DataFrame(split_cmp).T
split_val

**À commenter** : l'écart entre les deux estimations ; lequel est optimiste et pourquoi (fenêtres jumelles, même
personne). La réponse définitive (« lequel est le plus proche du test ? ») est donnée en section 7.

### 3.3 Référence : régression logistique multiclasse (P1.4)
Tous les réseaux seront comparés à cette référence (même découpage par volontaire, mêmes entrées standardisées).

In [ ]:
t0 = time.time()
baseline = LogisticRegression(max_iter=3000).fit(X_tr, y_tr)
baseline_time = time.time() - t0
yp_base = baseline.predict(X_va)
baseline_val = metrics.compute_metrics(y_va, yp_base)
print(f"référence — validation : accuracy {baseline_val['accuracy']:.4f} · F1 macro {baseline_val['f1_macro']:.4f}"
      f" · {baseline_time:.1f} s · {baseline.coef_.size + baseline.intercept_.size} paramètres")
display(pd.Series(metrics.per_class_f1(y_va, yp_base), name="F1 par classe (val)").to_frame())
plots.confusion_heatmap(metrics.confusion(y_va, yp_base), "Référence — validation", save="v1_baseline_confusion_val");

## 4. Définition du modèle et des fonctions d'entraînement / évaluation
`train_epoch`, `evaluate`, `fit`, `run_seeds` et `lr_range_test` sont dans `src/har/training.py` ; le MLP dans
`src/har/models.py`. Modèle de base : 561 → 64 → 6.

In [ ]:
m = models.MLP(IN_DIM, [64], activation="relu")
print(m)
print("nombre de paramètres :", models.count_params(m), "= 561×64 + 64 + 64×6 + 6")

### 4.1 Vérifications avant entraînement (P2.1)
**Perte initiale** : à l'initialisation, les sorties sont proches de 0, donc la softmax est quasi uniforme sur les
6 classes : on attend $-\ln(1/6) = \ln 6 \approx 1{,}792$.

In [ ]:
init_losses = []
for s in SEEDS:
    training.set_seed(s)
    m = models.MLP(IN_DIM, [64], activation="relu").to(training.DEVICE)
    init_losses.append(training.evaluate(m, val_loader, CRIT)["loss"])
print("perte initiale (3 graines) :", ms(init_losses), "| attendu :", round(np.log(6), 4))

**Surapprentissage d'un petit lot** : le réseau doit pouvoir apprendre par cœur 64 exemples (perte → 0,
accuracy → 100 %). Sinon, il y a un bug (données, perte, optimiseur).

In [ ]:
training.set_seed(0)
Xs, ys = next(iter(train_loader))
small = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xs, ys), batch_size=len(ys))
m = models.MLP(IN_DIM, [64], activation="relu")
h_small = training.fit(m, small, small, torch.optim.Adam(m.parameters(), lr=1e-2), epochs=50 if FAST else 300)
print(f"petit lot : perte {h_small['train_loss'][0]:.3f} → {h_small['train_loss'][-1]:.5f} ; "
      f"accuracy finale {h_small['train_acc'][-1]:.0%}")
plt.figure(figsize=(6, 3)); plt.semilogy(h_small["train_loss"]); plt.xlabel("itération"); plt.ylabel("perte (log)")
plt.title("Surapprentissage d'un lot de 64 exemples"); plt.tight_layout(); plt.savefig(config.FIG_DIR / "v1_overfit_batch.png", dpi=200)

## 5. Boucle d'entraînement principale
### 5.1 Réglage du pas d'apprentissage (P2.2)
**LR range test** (Smith, 2017) : on augmente le pas exponentiellement pendant ~100 itérations et on regarde où la perte
diminue le plus vite et où elle diverge. Il donne une plage ; la **grille logarithmique** ci-dessous tranche.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, act in zip(axes, ACTS):
    for opt_name in OPTS:
        training.set_seed(0)
        m = models.MLP(IN_DIM, [64], activation=act)
        opt = training.make_optimizer(opt_name, m.parameters(), lr=1e-6)
        lrs, losses = training.lr_range_test(m, train_loader, opt, lr_min=1e-5, lr_max=3, n_steps=120)
        smooth = pd.Series(losses).ewm(alpha=0.1).mean()
        ax.semilogx(lrs, smooth, label=opt_name)
    ax.set_title(f"LR range test — {act}"); ax.set_xlabel("pas d'apprentissage"); ax.set_ylim(0, 2.5); ax.legend()
axes[0].set_ylabel("perte (lissée)")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "v1_lr_range_test.png", dpi=200)

In [ ]:
# Grille logarithmique : chaque (activation, optimiseur, lr), graine 0, EPOCHS_LR époques.
# Critère : F1 macro de validation moyen sur les 3 dernières époques.
grid_rows = []
t0 = time.time()
for act in ACTS:
    for opt_name in OPTS:
        for lr in LR_GRID:
            training.set_seed(0)
            tl, vl = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
            m = models.MLP(IN_DIM, [64], activation=act)
            h = training.fit(m, tl, vl, training.make_optimizer(opt_name, m.parameters(), lr=lr), epochs=EPOCHS_LR)
            score = last_k(h) if np.isfinite(h["train_loss"][-1]) else 0.0
            grid_rows.append({"activation": act, "optimiseur": opt_name, "lr": lr, "val_f1": score,
                              "train_loss": h["train_loss"][-1]})
print(f"grille : {len(grid_rows)} entraînements en {time.time() - t0:.0f} s")
grid = pd.DataFrame(grid_rows)
grid.to_csv(config.LOG_DIR / "v1_lr_grid.csv", index=False)

BEST_LR = grid.loc[grid.groupby(["activation", "optimiseur"])["val_f1"].idxmax()] \
              .set_index(["activation", "optimiseur"])["lr"].to_dict()
grid.pivot_table(index=["activation", "optimiseur"], columns="lr", values="val_f1").round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, act in zip(axes, ACTS):
    for opt_name in OPTS:
        g = grid[(grid.activation == act) & (grid.optimiseur == opt_name)]
        ax.semilogx(g["lr"], g["val_f1"], "o-", label=f"{opt_name} (meilleur : {BEST_LR[(act, opt_name)]:g})")
    ax.set_title(act); ax.set_xlabel("pas d'apprentissage"); ax.legend(fontsize=8)
axes[0].set_ylabel(f"F1 macro val ({EPOCHS_LR} époques)")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "v1_lr_grid.png", dpi=200)
pd.Series({f"{a} / {o}": lr for (a, o), lr in BEST_LR.items()}, name="pas retenu").to_frame()

**À commenter** : cohérence entre le LR range test et la grille ; ordre de grandeur des pas retenus selon l'optimiseur
(SGD simple tolère / exige des pas plus grands qu'Adam) ; effet de l'activation (Sigmoid sature → pas plus grands ou
convergence plus lente).

### 5.2 Activations × optimiseurs, chacun avec son pas réglé, sur 3 graines (P2.2 et P2.3)
9 configurations × 3 graines × `EPOCHS` époques. Les métriques (loss, accuracy, F1 macro) sont suivies à chaque époque
sur l'apprentissage et la validation (historiques conservés pour les figures).

In [ ]:
def build_cfg(act, opt_name, lr, hidden=(64,)):
    def fn(seed):
        tl, vl = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=seed)
        m = models.MLP(IN_DIM, list(hidden), activation=act)
        return m, tl, vl, training.make_optimizer(opt_name, m.parameters(), lr=lr)
    return fn

runs = {}
t0 = time.time()
for act in ACTS:
    for opt_name in OPTS:
        runs[(act, opt_name)] = training.run_seeds(build_cfg(act, opt_name, BEST_LR[(act, opt_name)]),
                                                   seeds=SEEDS, keep_models=True, epochs=EPOCHS)
print(f"{len(runs) * len(SEEDS)} entraînements en {time.time() - t0:.0f} s")

def summarize(r):
    hs = [x["history"] for x in r["runs"]]
    return {
        "val acc": ms([last_k(h, "val_acc") for h in hs]),
        "val F1 macro": ms([last_k(h, "val_f1") for h in hs]),
        "train acc": ms([last_k(h, "train_acc") for h in hs]),
        "écart loss val−train": ms([h["val_loss"][-1] - h["train_loss"][-1] for h in hs]),
        "meilleure époque (F1 val)": ms([int(np.argmax(h["val_f1"])) + 1 for h in hs]),
        "temps (s)": ms([h["train_time_s"] for h in hs]),
        "_f1_mean": np.mean([last_k(h, "val_f1") for h in hs]),
        "_f1_std": np.std([last_k(h, "val_f1") for h in hs], ddof=1),
    }

NAME = {k: f"{k[0]} / {k[1]} (lr={BEST_LR[k]:g})" for k in runs}   # (activation, optimiseur) -> libellé
KEY = {v: k for k, v in NAME.items()}
table = pd.DataFrame({NAME[k]: summarize(r) for k, r in runs.items()}).T
table = table.sort_values("_f1_mean", ascending=False)
base_row = pd.DataFrame({"val acc": f"{baseline_val['accuracy']:.4f}", "val F1 macro": f"{baseline_val['f1_macro']:.4f}"},
                        index=["référence (régression logistique)"])
display(pd.concat([table.drop(columns=["_f1_mean", "_f1_std"]), base_row]))
table.to_csv(config.LOG_DIR / "v1_act_opt_seeds.csv")

### 5.3 Les écarts sont-ils plus grands que la variabilité ? (P2.3)
Règle du sujet : un écart entre deux configurations plus petit que la variabilité entre graines ne permet pas de
conclure. Critère simple utilisé ici : on conclut si |Δ moyenne| > écart-type₁ + écart-type₂.

In [ ]:
order = list(table.index)
stats = table[["_f1_mean", "_f1_std"]].astype(float)
best_name = order[0]
cmp_rows = []
for other in order[1:]:
    d = stats.loc[best_name, "_f1_mean"] - stats.loc[other, "_f1_mean"]
    tol = stats.loc[best_name, "_f1_std"] + stats.loc[other, "_f1_std"]
    cmp_rows.append({"comparé à": other, "Δ F1 macro": round(d, 4), "σ₁+σ₂": round(tol, 4),
                     "conclusion": "meilleure (écart > variabilité)" if d > tol else "non concluant"})
print("meilleure configuration (val) :", best_name)
d_base = stats.loc[best_name, "_f1_mean"] - baseline_val["f1_macro"]
print(f"écart avec la référence : {d_base:+.4f} (σ du MLP : {stats.loc[best_name, '_f1_std']:.4f})")
pd.DataFrame(cmp_rows)

### 5.4 Accuracy ou F1-score ?
L'accuracy compte les bonnes prédictions toutes classes confondues ; le F1 macro moyenne le F1 de chaque classe et
pénalise donc une classe mal reconnue même si elle est peu fréquente. Le F1 par classe montre *où* sont les erreurs.

In [ ]:
best_key = KEY[best_name]
best_runs = runs[best_key]
best_model = best_runs["runs"][0]["model"]
ev_best = training.evaluate(best_model, val_loader, CRIT, return_preds=True)
f1_cls = pd.DataFrame({
    "MLP retenu": metrics.per_class_f1(ev_best["y_true"], ev_best["y_pred"]),
    "référence": metrics.per_class_f1(y_va, yp_base),
    "part des fenêtres (val)": pd.Series(np.bincount(y_va, minlength=6) / len(y_va), index=config.CLASSES),
}).round(3)
display(f1_cls)
acc_f1 = pd.DataFrame({k: {"accuracy": last_k(r["runs"][0]["history"], "val_acc"),
                           "F1 macro": last_k(r["runs"][0]["history"], "val_f1")} for k, r in runs.items()}).T
print("corrélation accuracy / F1 macro entre les 9 configurations :", round(acc_f1.corr().iloc[0, 1], 3))
print("écart max |accuracy − F1 macro| :", round((acc_f1["accuracy"] - acc_f1["F1 macro"]).abs().max(), 4))

**À commenter** : avec des classes presque équilibrées (Partie 1.2), accuracy et F1 macro sont proches et classent les
configurations de la même façon ; le F1 par classe reste le plus informatif car il isole les classes difficiles
(SITTING / STANDING). Si les classes étaient déséquilibrées, l'accuracy pourrait masquer une classe rare mal prédite.

## 6. Visualisation et comparaison des résultats
### 6.1 Courbes d'apprentissage de la meilleure configuration (3 graines)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for r in best_runs["runs"]:
    h = r["history"]
    for ax, key, lab in zip(axes, ["loss", "acc", "f1"], ["Loss", "Accuracy", "F1 macro"]):
        ax.plot(h[f"train_{key}"], color="C0", alpha=0.7, label="train" if r["seed"] == SEEDS[0] else None)
        ax.plot(h[f"val_{key}"], color="C1", alpha=0.7, label="validation" if r["seed"] == SEEDS[0] else None)
        ax.set_title(lab); ax.set_xlabel("époque")
axes[0].legend(); fig.suptitle(f"MLP [64] — {best_name} — 3 graines")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "v1_best_learning_curves.png", dpi=200)

### 6.2 Comparaison des 9 configurations (graine 0)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for j, opt_name in enumerate(OPTS):
    for act in ACTS:
        h = runs[(act, opt_name)]["runs"][0]["history"]
        axes[0, j].plot(h["val_loss"], label=f"{act} val"); axes[0, j].plot(h["train_loss"], "--", alpha=0.6)
        axes[1, j].plot(h["val_f1"], label=act)
    axes[0, j].set_title(f"{opt_name} — loss (trait plein : val, pointillé : train)")
    axes[1, j].set_title(f"{opt_name} — F1 macro val"); axes[1, j].set_xlabel("époque")
    axes[0, j].legend(fontsize=8); axes[1, j].legend(fontsize=8)
plt.tight_layout(); plt.savefig(config.FIG_DIR / "v1_compare_configs.png", dpi=200)

In [ ]:
plots.confusion_heatmap(metrics.confusion(ev_best["y_true"], ev_best["y_pred"]), f"MLP [64] {best_name} — validation",
                        save="v1_best_confusion_val");

**À commenter** (convergence, stabilité, sous/sur-apprentissage) :
- vitesse de convergence selon l'optimiseur (Adam / momentum vs SGD simple) et l'activation (Sigmoid plus lente) ;
- stabilité : écart-type entre graines, oscillations des courbes de validation ;
- sur-apprentissage : la loss de validation remonte-t-elle alors que celle d'apprentissage baisse ? À partir de quelle
  époque ? L'accuracy d'apprentissage atteint-elle ~100 % ? ;
- comparaison à la référence : le MLP fait-il mieux, au-delà de la variabilité ?

In [ ]:
# sauvegarde de la configuration retenue (réutilisée par 06_interpretability.ipynb)
best_act, best_opt = best_key
selected = {"hidden": [64], "activation": best_act, "optimizer": best_opt, "lr": BEST_LR[best_key], "epochs": EPOCHS,
            "val_f1": table.loc[best_name, "val F1 macro"], "val_acc": table.loc[best_name, "val acc"],
            "val_subjects": [int(s) for s in val_subjects]}
training.save_json(selected, config.LOG_DIR / "v1_selected.json")
training.save_json({"baseline_val": baseline_val, "split_val": {f"{a} | {b}": v for (a, b), v in split_cmp.items()},
                    "best_lr": {f"{a}/{o}": lr for (a, o), lr in BEST_LR.items()}}, config.LOG_DIR / "v1_val_results.json")
selected

## 7. Évaluation finale sur le jeu de test
⚠️ **Exécutée une seule fois**, après avoir figé tous les choix ci-dessus. Sont évalués :
la référence, le MLP retenu (ses 3 graines → moyenne ± écart-type), et les modèles de la comparaison des découpages
(P1.3). La comparaison avec la V2 (Madjiguène) se fait dans le rapport.

In [ ]:
df_test = data.load_test()
X_te_raw, y_te = data.to_xy(df_test)
X_te = scaler.transform(X_te_raw).astype(np.float32)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.as_tensor(X_te), torch.as_tensor(y_te)), batch_size=512)
print(f"test : {len(df_test)} fenêtres, {df_test[config.SUBJECT_COL].nunique()} volontaires")

In [ ]:
# Référence et MLP retenu : validation vs test
yp_base_te = baseline.predict(X_te)
base_te = metrics.compute_metrics(y_te, yp_base_te)
mlp_te = [training.evaluate(r["model"], test_loader, CRIT, return_preds=True) for r in best_runs["runs"]]

final = pd.DataFrame({
    "référence (régression logistique)": {
        "val accuracy": f"{baseline_val['accuracy']:.4f}", "test accuracy": f"{base_te['accuracy']:.4f}",
        "val F1 macro": f"{baseline_val['f1_macro']:.4f}", "test F1 macro": f"{base_te['f1_macro']:.4f}"},
    f"MLP [64] {best_name}": {
        "val accuracy": table.loc[best_name, "val acc"], "test accuracy": ms([e["accuracy"] for e in mlp_te]),
        "val F1 macro": table.loc[best_name, "val F1 macro"], "test F1 macro": ms([e["f1_macro"] for e in mlp_te])},
}).T
display(final)

In [ ]:
# F1 par classe et matrices de confusion sur le test
f1_te = pd.DataFrame({
    "référence": metrics.per_class_f1(y_te, yp_base_te),
    "MLP retenu (graine 0)": metrics.per_class_f1(mlp_te[0]["y_true"], mlp_te[0]["y_pred"]),
}).round(3)
display(f1_te)
plots.confusion_heatmap(metrics.confusion(y_te, yp_base_te), "Référence — test", save="v1_baseline_confusion_test")
plots.confusion_heatmap(metrics.confusion(mlp_te[0]["y_true"], mlp_te[0]["y_pred"]),
                        f"MLP [64] {best_name} — test", save="v1_best_confusion_test");

### 7.1 Réponse à la P1.3 : quel découpage de validation prédit le mieux le test ?

In [ ]:
rows = []
for name, sm in split_models.items():
    Xt = sm["scaler"].transform(X_te_raw).astype(np.float32)
    lt = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.as_tensor(Xt), torch.as_tensor(y_te)), batch_size=512)
    test_lr = metrics.compute_metrics(y_te, sm["logreg"].predict(Xt))["accuracy"]
    test_mlp = np.mean([training.evaluate(m, lt, CRIT)["accuracy"] for m in sm["mlps"]])
    for model_name, test_acc in [("régression logistique", test_lr), ("MLP [64]", test_mlp)]:
        val_acc = split_cmp[(name, model_name)]["accuracy"]
        rows.append({"découpage": name, "modèle": model_name, "accuracy val": round(val_acc, 4),
                     "accuracy test": round(test_acc, 4), "val − test": round(val_acc - test_acc, 4)})
split_test = pd.DataFrame(rows)
display(split_test)

training.save_json({"final": final.to_dict(), "f1_per_class_test": f1_te.to_dict(),
                    "split_comparison": split_test.to_dict(orient="records")}, config.LOG_DIR / "v1_test_results.json")

**À commenter** :
- écart validation → test pour la référence et le MLP : la validation par volontaire est-elle une bonne estimation ?
- P1.3 : le découpage aléatoire surestime-t-il la performance (val − test plus grand) ? Conclure sur la méthodologie ;
- classes les plus confondues sur le test (SITTING ↔ STANDING attendu : même posture statique, le téléphone à la
  ceinture voit surtout la gravité, très proche dans les deux cas) ; WALKING_UPSTAIRS ↔ DOWNSTAIRS ;
- le MLP fait-il mieux que la référence sur le test, au-delà de l'écart-type ?